# 대학 푸드코트 학습 — 시간 제한 없이 이어서 실행
완료한 작업은 건너뛰고 저장된 학습 상태부터 남은 작업을 끝까지 실행합니다. 기존 실험 ID와 학습·평가 조건은 유지하며 매 세대 Google Drive에 저장합니다.
프로그램에 설정한 12시간 제한을 제거했습니다. Colab 자체에서 런타임이 종료되면 이 노트북을 다시 실행해 저장된 지점부터 이어갈 수 있습니다.

In [ ]:
import os, subprocess, pathlib
def run_visible(args, cwd=None):
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status=process.wait()
    if status:
        raise subprocess.CalledProcessError(status,args)

os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
REPOSITORY="https://github.com/phj9187/koisk-simulation.git"
REVISION="7d677f76ed8e57a986f865db55de6a054f116be9" # 검증한 알고리즘 버전 고정
WORK=pathlib.Path("/content/foodcourt_code")
if not WORK.exists(): run_visible(["git","clone",REPOSITORY,str(WORK)])
run_visible(["git","-C",str(WORK),"fetch","origin",REVISION])
run_visible(["git","-C",str(WORK),"checkout","--detach","FETCH_HEAD"])
run_visible(["git","-C",str(WORK),"rev-parse","HEAD"])
run_visible(["pip","install","-q","-r",str(WORK/"requirements-foodcourt.txt")])
os.environ["PYTHONPATH"]=str(WORK/"src")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
USE_DRIVE=True
BASE=pathlib.Path("/content/drive/MyDrive/kiosk_foodcourt")
PRESET="colab"
EXPERIMENT_ID="foodcourt_research_20261103_001"
ROOT=BASE/EXPERIMENT_ID
CONFIG=WORK/"config"/"foodcourt_colab.yaml"
command=["python","-u","-m","kiosk_foodcourt.cli","--config",str(CONFIG),"--root",str(ROOT)]
print(subprocess.check_output(command+["--plan"],cwd=WORK,text=True))
print("결과와 학습 상태 저장 위치:",ROOT)


다음 셀은 모든 작업이 완료될 때까지 실행합니다. 시간 제한은 없으며 진행률과 저장된 학습 세대를 표시합니다. Colab 자체의 연결·자원 제한은 별도로 적용됩니다. 다른 탭에서 같은 실험을 중복 실행하지 마세요.

In [ ]:
import json, time, signal
from datetime import datetime, timezone, timedelta
previous=globals().get("training_process")
if previous is not None and previous.poll() is None:
    raise RuntimeError("이미 학습이 실행 중입니다. 중복 실행하지 않습니다.")
if not pathlib.Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive가 연결되지 않았습니다. 연결 셀을 먼저 실행하세요.")
ROOT.mkdir(parents=True,exist_ok=True)
LOG=ROOT/"training_until_complete.log"
launch_args=["python","-u",str(WORK/"tools"/"run_foodcourt_until_complete.py"),"--config",str(CONFIG),"--root",str(ROOT)]
revision=subprocess.check_output(["git","-C",str(WORK),"rev-parse","HEAD"],text=True).strip()
launch_record={"experiment":EXPERIMENT_ID,"source_commit":revision,"command":launch_args,"time_limit":None,"mode":"until_complete","started_at_kst":datetime.now(timezone(timedelta(hours=9))).isoformat()}
(ROOT/"launch_resume_20261104.json").write_text(json.dumps(launch_record,ensure_ascii=False,indent=2))
(ROOT/"research_config.yaml").write_text(CONFIG.read_text())
print("대규모 학습 시작:",EXPERIMENT_ID,flush=True)
print("240개 조건, 각 조건의 고정형·상황형 100세대, 2개 작업 병렬",flush=True)
print("프로그램 시간 제한 없이 남은 작업을 끝까지 처리합니다. 완료한 세대는 Drive에 저장합니다.",flush=True)
print("전체 실행 로그:",LOG,flush=True)
last_progress=None
with LOG.open("a",buffering=1) as log:
    training_process=subprocess.Popen(launch_args,cwd=WORK,stdout=log,stderr=subprocess.STDOUT,text=True,start_new_session=True)
    print("학습 프로세스 PID:",training_process.pid,flush=True)
    try:
        while training_process.poll() is None:
            completed=len(list((ROOT/"jobs").glob("*/result.json")))
            generations=0
            candidate_calls=0
            for file in (ROOT/"jobs").glob("*/checkpoint.json"):
                try:
                    state=json.loads(file.read_text())
                    generations+=state["generation"]
                    candidate_calls+=sum(model["calls"] for model in state["models"].values())
                except (OSError,ValueError,KeyError):
                    continue
            progress=(completed,generations,candidate_calls)
            if progress!=last_progress:
                status={"experiment":EXPERIMENT_ID,"status":"running","completed_jobs":completed,"total_jobs":240,"saved_generations":generations,"target_generations":24000,"candidate_training_calls":candidate_calls,"pid":training_process.pid,"updated_at_kst":datetime.now(timezone(timedelta(hours=9))).isoformat()}
                temp=ROOT/"training_status.tmp"
                temp.write_text(json.dumps(status,ensure_ascii=False,indent=2))
                os.replace(temp,ROOT/"training_status.json")
                print(f"완료 작업 {completed}/240 | 저장된 학습 세대 {generations}/24000 | 후보 학습 실행 {candidate_calls:,}회",flush=True)
                last_progress=progress
            time.sleep(30)
    except KeyboardInterrupt:
        if training_process.poll() is None:
            os.killpg(training_process.pid,signal.SIGTERM)
            training_process.wait()
        print("사용자 중단. 저장된 세대까지의 학습 상태는 Drive에 남습니다.",flush=True)
        raise
status_code=training_process.returncode
completed=len(list((ROOT/"jobs").glob("*/result.json")))
print("현재 실행 종료 코드:",status_code,"완료 작업:",completed,"/240",flush=True)
print(LOG.read_text()[-4000:],flush=True)
status={"experiment":EXPERIMENT_ID,"status":"complete" if completed==240 and status_code==0 else "incomplete" if status_code==0 else "failed","completed_jobs":completed,"total_jobs":240,"exit_code":status_code,"updated_at_kst":datetime.now(timezone(timedelta(hours=9))).isoformat()}
(ROOT/"training_status.json").write_text(json.dumps(status,ensure_ascii=False,indent=2))
if status_code:
    raise subprocess.CalledProcessError(status_code,launch_args)

if completed!=240:
    raise RuntimeError("전체 작업이 아직 완료되지 않았습니다. 로그와 저장된 학습 상태를 확인하세요.")


In [ ]:
import json
summary=ROOT/"summary.json"
completed=len(list((ROOT/"jobs").glob("*/result.json")))
print(f"평가까지 완료한 작업: {completed}/240")
if summary.exists():
    rows=json.loads(summary.read_text())
    for row in rows:
        if row["policy"].startswith("learned") and row["baseline"]=="customer_only" and row["scenario"]=="final_test":
            print(f"메뉴 {row['menu_count']} / {row['attention']} / 방문 평균 {row['visitors']} / {row['policy']} | 손님 점수 변화 {row['customer_change']:+.4f} | 공헌이익 변화 {row['profit_change_won']:+,.0f}원 | 독립 캠퍼스 {row['independent_courts']}개")
print("결과와 학습 상태:",ROOT)
print("실제 학교 관측 결과가 아닌 가상 환경의 결과입니다. 집단별 손실과 스트레스 평가도 함께 확인하세요.")
